# ViPRA — Qwen2.5-3B-Instruct, thực nghiệm learning rate và giới hạn sinh

Backbone **3,09B**, LoRA FP16 + **FSDP2 song song trên T4 x2**. Bật Internet và Add Input chứa hai CSV.
Clone nhánh `Nhotest` vào `/kaggle/working/VLSP`:

```bash
git clone --branch Nhotest --single-branch https://github.com/NgocNhoNguyen2605n/VLSP.git /kaggle/working/VLSP
```

Chạy lần lượt 6 cấu hình: LR {1e-3, 1e-4, 2e-4} × max_new_tokens {300, 1000}, weight_decay=0.1.
Mỗi cấu hình học toàn bộ train và chấm dev trong 5 epoch, cùng seed 42, khởi tạo model/LoRA mới.
Xuất best adapter mỗi cấu hình, chọn best trial theo điểm dev. Không retrain.


In [ ]:
import os  # Thiết lập môi trường cho các tiến trình train/evaluate.
import subprocess  # Chạy Python riêng để trả lại VRAM sau mỗi tác vụ.
import sys  # Dùng đúng Python của notebook Kaggle.
from pathlib import Path  # Đường dẫn repo, Kaggle Input và kết quả.

REPO = Path("/kaggle/working/VLSP")
DATA_ROOT = "/kaggle/input"
SWEEP_DIR = "/kaggle/working/vipra_qwen25_3b_lr_tokens01"  # Thư mục mới cho mỗi lưới.
RESUME_SWEEP = False  # True khi tiếp tục lưới dở sau khi restart session.
assert (REPO / "requirements.txt").is_file(), "Hãy clone repo vào /kaggle/working/VLSP trước."

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8",
                  TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1",
                  HF_HOME="/tmp/vipra_hf_cache")  # Không lưu backbone vào output notebook.

def run(module, *options):
    "Chạy tác vụ trong repo, dừng notebook nếu có lỗi."
    subprocess.run([sys.executable, "-m", module, *options], cwd=REPO, check=True)

# Gỡ torchao cũ của Kaggle: PEFT kiểm tra cả khi chỉ dùng LoRA FP16.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                "torchao"], check=True)

# Cài dependency; cần PyTorch CUDA >= 2.8 cho FSDP2.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "requirements.txt")], check=True)


## 1. Kiểm tra dữ liệu và template

Tự tìm CSV trong Input. Nếu có nhiều bản cùng tên, thu hẹp `DATA_ROOT`
hoặc truyền `--train-file` và `--dev-file` cụ thể vào cả check/train.
Không tải trọng số backbone trong bước này.


In [ ]:
# Kiểm tra các mẫu và mask assistant; mẫu dài sẽ báo lỗi thay vì bị cắt.
run("baseline.check", "--data-root", DATA_ROOT)


## 2. Chạy tuần tự 6 cấu hình, mỗi cấu hình 5 epoch

**676 mẫu train hợp lệ / 339 mẫu dev validation**, seed 42. Train dùng toàn bộ train_dataset.csv;
LD111 thiếu answer được bỏ theo cấu hình. Dev chỉ dùng chấm loss và sinh đáp án, không cập nhật gradient.

Lưới ở configs/sweep.json: learning_rates=[1e-3, 1e-4, 2e-4], weight_decays=[0.1], max_new_tokens=[300, 1000].
Mỗi trial train 5 epoch từ backbone gốc + LoRA mới, tổng 30 epoch training. Trial sau bắt đầu khi trial trước kết thúc.
Sau mỗi epoch, validation greedy theo batch 4 và mức token của trial, lưu checkpoint + predictions + metrics.
EOS/deadline 60 giây mỗi batch có thể kết thúc trước 300/1000 token. Training không cắt CoT theo mức token này.
Chọn content_tolerant accuracy cao nhất; nếu hòa: exact, loss thấp hơn, epoch sớm hơn.
Kết thúc 5 epoch, xuất đúng checkpoint thắng vào **thư mục trial/adapter**, không retrain.

Batch train 4/GPU × 2 GPU, tích lũy 2 = effective batch 16. Có **85 batch train/epoch** và
**85 batch generation validation/epoch**. Max_length 2048, LoRA r=16/alpha=32, backbone FP16.

Launcher dùng torchrun 2 worker FSDP2. Hai rank cùng sinh một batch validation, chỉ rank 0 ghi điểm.
Decoder frozen shard; embedding/head tied và LoRA replicate. Gradient LoRA all-reduce sau backward;
activation checkpoint bên trong FSDP, backward prefetch tắt. Preflight CUDA/NCCL phải PASS trước khi nạp 3B.

Tiến độ hiển thị batch và ETA của pha hiện tại. Với output pipe/Save Version, mỗi bản cập nhật
có xuống dòng: mỗi 10 batch hoặc sau 30 giây ở batch vừa hoàn tất, luôn in batch đầu/cuối.
ETA không tính validation hay checkpoint sắp tới. verbose_logging=false ẩn bảng thông tin chi tiết.

Chọn SWEEP_DIR mới mỗi lưới. Nếu NCCL abort, restart session rồi đặt RESUME_SWEEP=True để tiếp tục.
Trial hoàn tất được bỏ qua; trial lỗi chạy lại ở thư mục attempt mới, không ghi đè log cũ.
rank-0-progress.json và rank-1-progress.json trong thư mục trial/search ghi pha cuối của mỗi worker.


In [ ]:
# 6 trial tuần tự, mỗi trial train 5 epoch, validation dev, lưu best adapter.
options = ["--data-root", DATA_ROOT, "--output-dir", SWEEP_DIR]
if RESUME_SWEEP:
    options.append("--resume")
run("baseline.sweep", *options)


## 3. Xem bảng kết quả và best trial

results.csv tổng hợp các cấu hình. sweep.json lưu best trial.
Best adapter của trial nằm ở thư mục trial/adapter, không cần retrain hay đánh giá lại dev.
Dev đã dùng chọn checkpoint/cấu hình, nên điểm này là validation, không phải test độc lập.


In [ ]:
import csv, json

summary = json.loads((Path(SWEEP_DIR) / "sweep.json").read_text())
with (Path(SWEEP_DIR) / "results.csv").open() as handle:
    for row in csv.DictReader(handle):
        print(f"lr={row['learning_rate']} | wd={row['weight_decay']} | tokens={row['max_new_tokens']} | "
              f"epoch={row['best_epoch']} | đúng={row['content_tolerant']}/{row['total']} | {row['status']}")
best_trial = summary["best_trial"]
RUN_DIR = best_trial["run_dir"]
best = best_trial["best"]
print("Best trial:", best_trial["name"], "| Best epoch:", best["epoch"])
print("Adapter:", str(Path(RUN_DIR) / "adapter"))
print("Predictions dev:", str(Path(RUN_DIR) / "search" / "validation" / f"epoch-{best['epoch']}" / "predictions.jsonl"))


## 4. Đánh giá lại best adapter (tùy chọn)

Mặc định bỏ qua để tiết kiệm thời gian vì dev đã được chấm sau mỗi epoch.
Nếu cần đo thời gian từng câu/P1 ước lượng, bật RUN_EXTRA_EVALUATION.
Lệnh evaluate dùng RUN_DIR/adapter của best epoch và sinh một câu/lượt; không retrain.


In [ ]:
RUN_EXTRA_EVALUATION = False
if RUN_EXTRA_EVALUATION:
    run("baseline.evaluate", "--run-dir", RUN_DIR)
else:
    print("Dùng điểm dev của best epoch đã lưu; bỏ qua đánh giá lại.")


Lưới: configs/sweep.json. Tổng hợp: SWEEP_DIR/results.csv và SWEEP_DIR/sweep.json.
Mỗi trial: split.json, selection.json, search/ (checkpoint + validation), adapter/ của **best epoch**.
Lệnh evaluate tùy chọn dùng RUN_DIR của best trial và lưu RUN_DIR/evaluation/.
Muốn đánh giá lại dùng --result-dir mới để giữ kết quả cũ.
